# v6 Q2 run (GPU 1): Hindi only + a push at every phase boundary, from phase 30
Hindi-only run `v4b_only_hi_s0` continues from phase 30, still only Hindi. At the start of every phase, noise of 0.6 x each weight matrix's std is added, then the phase trains as usual. So the network moves about as much as at a language switch, but never switches. Question (Q2): is the self's behaviour at switches special to switching, or does any big push give the same?

**Just Run All.** Output appears below, as in v5. Rerunning resumes from the last saved phase.

In [1]:
import os, sys, glob, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "1"     # run this cell FIRST in a fresh kernel
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
os.system("pkill -f 'train_v6.py.*--gpu 1'")  # stop any background job from the earlier notebooks on this GPU
from common_v6 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")
RUN = "v6_hi_push0.6_from30"

device: cuda | NVIDIA GeForce RTX 4090


In [2]:
# get what this run needs (skips what is already here)
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):                          # retry: HF sometimes drops the connection
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
# 1) the checkpoint we branch from
SRC_RUN, SRC_PHASE = "v4b_only_hi_s0", 30
if not glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt"):
    name = next(f for f in api.list_repo_files(HF_REPO) if f.startswith(f"{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_"))
    hf_hub_download(HF_REPO, name, local_dir="runs")
INIT = glob.glob(f"runs/{SRC_RUN}/ckpt/p{SRC_PHASE:03d}_*.pt")[0]
print("branch from:", INIT)
# 2) a leftover folder from the earlier (crashed) attempt with nothing trained yet -> start clean
done = [p for p in glob.glob(f"runs/{RUN}/ckpt/p*.pt") if int(os.path.basename(p)[1:4]) > SRC_PHASE]
if os.path.isdir(f"runs/{RUN}") and not done:
    import shutil; shutil.rmtree(f"runs/{RUN}"); print("removed empty leftover folder")
# 3) FLORES dev + devtest
from datasets import load_dataset
for l, code in {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}.items():
    for split in ["dev", "devtest"]:
        p = f"data/flores/{split}.{l}.jsonl"
        if not os.path.exists(p):
            os.makedirs("data/flores", exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for t in ds["text"]:
                    f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
# 4) training data must be here
need = [f"data/en-{l}/train.jsonl" for l in ["hi"]] + [f"data/en-{l}/val.jsonl" for l in LANGS]
missing = [p for p in need if not os.path.exists(p)]
print("data OK" if not missing else f"MISSING: {missing}")

branch from: runs/v4b_only_hi_s0/ckpt/p030_hi.pt
removed empty leftover folder
data OK


In [3]:
C = RunConfig(run_name=RUN, mode="only_hi", order=("hi", "zh", "ar"), init_ckpt=INIT, start_phase=SRC_PHASE,
              n_phases=90, seed=0, time_cap_h=120.0, hf_repo=HF_REPO, boundary_push=0.6)
print(C)

RunConfig(run_name='v6_hi_push0.6_from30', mode='only_hi', order=('hi', 'zh', 'ar'), init_ckpt='runs/v4b_only_hi_s0/ckpt/p030_hi.pt', n_phases=90, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints', start_phase=30, noise_per_step=0.0, boundary_push=0.6, health_every=1)


In [4]:
D = load_data(train_langs=["hi"])

  hi: 993,016 train / 2,000 val
  zh: 2,000 val
  ar: 2,000 val
data loaded in 19s


In [ ]:
model = train_run(C, D)

 RUN v6_hi_push0.6_from30 | mode only_hi | 90 phases after phase 0 | device cuda
 phase order: init (loaded) → hi → hi → hi → hi → hi → hi → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
BRANCH: starting from runs/v4b_only_hi_s0/ckpt/p030_hi.pt (phase 30, hi); next phase 31

 v6: noise_per_step = 0.0 (x each matrix's starting std) | boundary_push = 0.6 (x each matrix's current std, at the start of every phase)

▶ PHASE 31/90 | training hi (993,016 pairs)
  push: added noise 0.6 x std to every weight matrix


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[W1026 22:35:30.100685651 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2048917504 bytes (free: 964427776, total: 25250627584).


Validating:           |   0%            

...

New Data Upload: |          |  0.00B /  0.00B            

  (HF backup failed, training continues: Previous task error: Network error: Request middleware error: error sending request for url (https://huggingface.co/api/models/s8n29/cml-lang-self-checkpoints/xet-write-token/main))
  [p54 hi] step    500 | lr 2.0e-04 | train 3.714 | val 1.756 (best 1.756) | window Δ n/a | stable | 75k tok/s | phase 1m37s | run ETA ~23h02m | health {'wnorm': 445.071, 'rank_enc1': 937.5, 'rank_dec1': 734.7, 'rarely_active': 0.029}
  [p54 hi] step  1,000 | lr 2.0e-04 | train 2.699 | val 1.508 (best 1.508) | window Δ n/a | stable | 77k tok/s | phase 3m12s | run ETA ~23h02m | health {'wnorm': 441.788, 'rank_enc1': 929.2, 'rank_dec1': 750.6, 'rarely_active': 0.0308}
  [p54 hi] step  1,500 | lr 2.0e-04 | train 2.511 | val 1.423 (best 1.423) | window Δ n/a | stable | 77k tok/s | phase 4m47s | run ETA ~23h02m | health {'wnorm': 438.603, 'rank_enc1': 915.8, 'rank_dec1': 745.8, 'rarely_active': 0.0319}
  [p54 hi] step  2,000 | lr 2.0e-04 | train 2.397 | val 1.391 (best 1.

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  (HF backup failed, training continues: Previous task error: Network error: Request middleware error: error sending request for url (https://huggingface.co/api/models/s8n29/cml-lang-self-checkpoints/xet-write-token/main))
  [p55 hi] step    500 | lr 2.0e-04 | train 3.729 | val 1.757 (best 1.757) | window Δ n/a | stable | 76k tok/s | phase 1m36s | run ETA ~22h24m | health {'wnorm': 449.769, 'rank_enc1': 932.4, 'rank_dec1': 753.8, 'rarely_active': 0.0269}
  [p55 hi] step  1,000 | lr 2.0e-04 | train 2.705 | val 1.514 (best 1.514) | window Δ n/a | stable | 77k tok/s | phase 3m11s | run ETA ~22h24m | health {'wnorm': 446.428, 'rank_enc1': 927.5, 'rank_dec1': 763.4, 'rarely_active': 0.0303}
  [p55 hi] step  1,500 | lr 2.0e-04 | train 2.520 | val 1.433 (best 1.433) | window Δ n/a | stable | 77k tok/s | phase 4m47s | run ETA ~22h24m | health {'wnorm': 443.176, 'rank_enc1': 919.2, 'rank_dec1': 743.9, 'rarely_active': 0.034}
  [p55 hi] step  2,000 | lr 2.0e-04 | train 2.407 | val 1.404 (best 1.

In [ ]:
print_run_summary(RUN)